In [ ]:
from tablevault import tablevault
import os
vault = tablevault.Vault(user_id="jinjin",
                            process_name="distilbert_feature_extraction_cosine_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [1]:
import torch
import numpy as np
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)


device: mps


In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [3]:
model_name = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print('hidden_size:', model.config.hidden_size)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds['sentence1']
sent2 = ds['sentence2']
y_true = np.array(ds['label'])

print('num_examples:', len(y_true))
print('positive_rate:', float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    pooled = summed / counts
    pooled = F.normalize(pooled, p=2, dim=1)
    return pooled


def encode_sentences(sentences, batch_size=128, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size)):
            batch = sentences[i:i + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            pooled = mean_pool(outputs.last_hidden_state, enc['attention_mask'])
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0)


In [6]:
emb1 = encode_sentences(sent1)
emb2 = encode_sentences(sent2)

vault.create_embedding_list("distilbert-feature-embedding-sentence-1", ndim=768)
vault.create_embedding_list("distilbert-feature-embedding-sentence-2", ndim=768)

e1_list = emb1.tolist()
e2_list = emb2.tolist()

for i in range(len(e1_list)):
        vault.append_embedding("distilbert-feature-embedding-sentence-1", e1_list[i], 
                           input_items = {"glue_mrpc_validation": [i, i + 1]}
                           )
        vault.append_embedding("distilbert-feature-embedding-sentence-2", e2_list[i], 
                           input_items = {"glue_mrpc_validation": [i, i + 1]}
                           )
    
description = "INSERT TEXT HERE ABOUT distilbert-feature-embedding-sentence-1"
embedding = get_embeddings(description)
vault.create_description("distilbert-feature-embedding-sentence-1", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-feature-embedding-sentence-1", cat, embedding, prop)

description = "INSERT TEXT HERE ABOUT distilbert-feature-embedding-sentence-2"
embedding = get_embeddings(description)
vault.create_description("distilbert-feature-embedding-sentence-2", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-feature-embedding-sentence-2", cat, embedding, prop)


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

done
embedding_shape: (408, 768)
score_range: 0.7408234477043152 0.9976521134376526


In [ ]:
cosine_scores = (emb1 * emb2).sum(dim=1).numpy()
threshold = 0.85
y_pred = (cosine_scores >= threshold).astype(int)

print('done')
print('embedding_shape:', tuple(emb1.shape))
print('score_range:', float(cosine_scores.min()), float(cosine_scores.max()))

vault.create_record_list("distilbert-feature-embedding_prediction", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("distilbert-feature-embedding_prediction", {"prediction": y_pred[i]}, 
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                           "distilbert-feature-embedding-sentence-1": [i, i + 1],
                           "distilbert-feature-embedding-sentence-2": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT distilbert-feature-embedding_prediction"
embedding = get_embeddings(description)
vault.create_description("distilbert-feature-embedding_prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-feature-embedding_prediction", cat, embedding, prop)

In [7]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])
print({'accuracy': acc, 'f1': f1})
print(classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase']))


{'accuracy': 0.696078431372549, 'f1': 0.8165680473372781}
                precision    recall  f1-score   support

not_paraphrase       0.73      0.06      0.11       129
    paraphrase       0.70      0.99      0.82       279

      accuracy                           0.70       408
     macro avg       0.71      0.53      0.47       408
  weighted avg       0.71      0.70      0.59       408



In [8]:
for i in range(5):
    print('=' * 80)
    print('sentence1:', sent1[i])
    print('sentence2:', sent2[i])
    print('cosine:', float(cosine_scores[i]))
    print('true:', int(y_true[i]), 'pred:', int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print('num_errors:', int((y_true != y_pred).sum()))

for i in mistakes:
    print('=' * 80)
    print('idx:', int(i))
    print('sentence1:', sent1[i])
    print('sentence2:', sent2[i])
    print('cosine:', float(cosine_scores[i]))
    print('true:', int(y_true[i]), 'pred:', int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
cosine: 0.9466804265975952
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
cosine: 0.8935593366622925
true: 0 pred: 1
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
cosine: 0.96232008934021
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will de

In [9]:
vault.create_record_list("distilbert_feature_extraction_cosine_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("distilbert_feature_extraction_cosine_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert-feature-embedding_prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT distilbert_feature_extraction_cosine_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("distilbert_feature_extraction_cosine_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_feature_extraction_cosine_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'distilbert-base-uncased',
 'device': 'mps',
 'threshold': 0.85,
 'num_examples': 408,
 'accuracy': 0.696078431372549,
 'f1': 0.8165680473372781}

In [ ]:
description = "INSERT TEXT HERE ABOUT distilbert_feature_extraction_cosine_mrpc process" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("distilbert_feature_extraction_cosine_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_feature_extraction_cosine_mrpc", cat, embedding, prop)